In [ ]:
# import libraries
import ollama
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma



In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
## Lets read the pdf files and split them into chunks
def read_doc(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

In [ ]:
doc = read_doc("documents/")
len(doc)

In [ ]:
## Devide the docs into chunks
def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc = text_splitter.split_documents(docs)
    return docs

In [ ]:
documents = chunk_data(docs=doc)
len(documents)

In [ ]:
print("Syncing with Docker Ollama...")
ollama.pull("nomic-embed-text")
ollama.pull("llama3")

embeddings = OllamaEmbeddings(model="nomic-embed-text")
print("✅ Setup Complete")

In [ ]:
# --- CONFIGURATION ---
PDF_PATH = "./documents/Vitamin_and_mineral_requirements.pdf"  # Put your PDF file name here
DB_DIR = "./my_local_db"
EMBED_MODEL = "nomic-embed-text"

In [ ]:
loader = PyPDFLoader(PDF_PATH)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = loader.load_and_split(text_splitter)

In [ ]:
# Create Local Vector DB
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./my_local_db"
)
print(f"✅ Success! {len(chunks)} chunks saved to disk.")

In [ ]:
query = "Tell me about this document. Whatever you understand." # 👈 Ask anything!

# 1. Search the PDF for relevant text
results = vector_store.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in results])

# 2. Get answer from Llama 3
response = ollama.chat(model="llama3", messages=[
    {"role": "system", "content": "You are a helpful assistant. Answer the question using ONLY the provided context from the PDF."},
    {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"}
])

print(f"\n🤖 AI ANSWER:\n{response['message']['content']}")


In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

# 1. Initialize local embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Load and split documents (300 token/character size, 30 overlap)
loader = PyPDFLoader("who_guideline.pdf")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(raw_docs)

# 3. Define fast deterministic filtering rules
# High-value clinical and biochemical keywords
MEDICAL_KEYWORDS = {
    "vitamin", "deficiency", "retinol", "phylloquinone", "gla", "osteocalcin", 
    "plasma", "intake", "requirement", "mg/day", "μg/day", "dietary", 
    "supplement", "serum", "clinical", "endpoint", "repletion", "adults"
}

# Clues that a chunk is just boilerplate, a bibliography, or a index
JUNK_PATTERNS = [
    r"isbn\s\d+",                     # Copyright info
    r"published\sby",                 # Publisher boilerplate
    r"pp\.\s\d+–\d+",                 # Citation page ranges
    r"doi:\s10\.",                    # Digital Object Identifiers
    r"contents\s+\d+",                # Table of contents lines
    r"^\s*\d+\s*$"                    # Standalone page numbers
]

important_documents = []

for chunk in chunks:
    text = chunk.page_content
    text_lower = text.lower()
    
    # Rule A: Skip pure junk/citations using fast regex
    if any(re.search(pattern, text_lower) for pattern in JUNK_PATTERNS):
        continue
        
    # Rule B: Calculate keyword density (Count how many unique target words appear)
    words_found = [word for word in MEDICAL_KEYWORDS if word in text_lower]
    
    # Rule C: Strict structural check (Skip chunks that are mostly numbers/symbols like big tables)
    alpha_chars = sum(c.isalpha() for c in text)
    total_chars = len(text) if len(text) > 0 else 1
    alpha_ratio = alpha_chars / total_chars
    
    # Triage: Keep it only if it contains at least 2 unique core keywords 
    # AND is mostly actual prose text (not a raw data table)
    if len(words_found) >= 2 and alpha_ratio > 0.60:
        # Prepend Nomic search prefix for best vector matching performance
        chunk.page_content = f"search_document: {chunk.page_content}"
        important_documents.append(chunk)

print(f"Instantly filtered {len(chunks)} down to {len(important_documents)} important chunks.")
